# Motion-Phase ML for Track & Field Athletes: End-to-End Workflow

## Notebook Guide
- **Part 1: Dataset Overview:** quick health checks, class balance, and sensor visualizations.
- **Part 2: Preprocessing:** duplicate removal, timestamp cleanup, sampling-rate diagnostics, and segmentation.
- **Part 3: Feature Engineering:** engineered statistics and spectral features, feature ranking, and artifact generation for modeling.
- **Part 4: Model Training and Hyperparameter Tuning:** model selection, pipeline setup, cross-validation, GridSearchCV tuning, and saving trained models.
- **Part 5:Evaluation:** 

## Setup and Configuration

This section imports all required libraries and defines global settings used throughout the notebook. It covers:
- Data handling and plotting libraries for analysis and diagnostics.
- Signal processing and feature-selection tools .
- Reproducible output paths and constants.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.fft import rfft, rfftfreq
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

plt.style.use("seaborn-v0_8")
sns.set_palette("crest")


In [ ]:
DATA_PATH = Path("data/biosensor_dataset_with_target.csv")
print(f"Using dataset at: {DATA_PATH.resolve()}")

### Runtime Output Paths & Parameters

In [ ]:
# root
OUT_DIR = Path("out") 

# section dirs
S1 = OUT_DIR / "01_dataset_exploration"
S2 = OUT_DIR / "02_signal_exploration"
S3 = OUT_DIR / "03_preprocessing"
S4 = OUT_DIR / "04_windowing"
S5 = OUT_DIR / "05_features"
S6 = OUT_DIR / "06_modeling"
S7 = OUT_DIR / "07_evaluation"

# leaf dirs
PATHS = [
    S1 / "tables", S1 / "plots",
    S2 / "plots",
    S3 / "plots",
    S4 / "tables", S4 / "plots",
    S5 / "tables", S5 / "plots",
    S6 / "tables", S6 / "plots",
    S7 / "tables", S7 / "plots",
    OUT_DIR / "report" / "figures",
]

for p in PATHS:
    p.mkdir(parents=True, exist_ok=True)

print(f"Artifacts will be saved under {OUT_DIR.resolve()}")

In [ ]:
RAW_SENSOR_COLS = [
    "Heart_Rate", "Acc_X", "Acc_Y", "Acc_Z",
    "Gyro_X", "Gyro_Y", "Gyro_Z",
]
FEATURE_COLS = RAW_SENSOR_COLS  # 7 channels: Heart_Rate + Acc_X/Y/Z + Gyro_X/Y/Z

SENSOR_THRESHOLDS = {
    "Heart_Rate": (60.0, 190.0),
    "Acc_X": (-3.2, 3.2), "Acc_Y": (-3.2, 3.2), "Acc_Z": (-3.2, 3.2),
    "Gyro_X": (-190.0, 190.0), "Gyro_Y": (-190.0, 190.0), "Gyro_Z": (-190.0, 190.0),
}

WINDOW_SECONDS = 1.0    # seconds — selected from sweep
FS             = 10.0    # Hz — confirmed from sampling rate analysis
STEP_RATIO     = 0.25
STEP_SECONDS   = WINDOW_SECONDS * STEP_RATIO
TOP_K          = 25
RANDOM_STATE   = 42

# Shared CV helpers, used by all model/sweep cells
le = LabelEncoder()

def make_rf():
    return RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

def run_rf_cv(X, y_enc, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(make_rf(), X, y_enc, cv=cv, scoring="f1_macro")

def safe_n_splits(y_enc, max_splits=5):
    return min(max_splits, np.bincount(y_enc).min())


## PART 1: DATASET OVERVIEW PORTION

### Read Raw Data from File

This cell loads the biosensor CSV into a dataframe and converts `Timestamp` into a proper datetime column.

It gives a first check that:
- the file path is correct,
- the number of rows and columns
- number of athletes and days per athlete
- demographic diversity check  


The printed summary and preview help confirm the raw file was read correctly.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw["Timestamp"] = pd.to_datetime(df_raw["Timestamp"], errors="coerce")
print(f"Reading from {DATA_PATH}")
print(f"Rows: {len(df_raw):,}  |  Columns: {len(df_raw.columns)}")

In [ ]:
n_athletes = df_raw["Athlete_ID"].nunique()
total_records = len(df_raw)
records_per_athlete_avg = total_records // n_athletes
sensor_channels = len(RAW_SENSOR_COLS)
n_classes = df_raw["Event_Label"].nunique()

# Data span per athlete
span_per_athlete = df_raw.groupby("Athlete_ID")["Timestamp"].apply(
    lambda t: (t.max() - t.min()).total_seconds()
)
avg_span_s = round(span_per_athlete.mean(), 1)

summary = pd.DataFrame({
    "Property": [
        "Number of athletes",
        "Total records",
        "Records per athlete (avg)",
        "Sensor channels",
        "Motion phase classes",
        "Data span per athlete (avg)",
        "Demographic info",
    ],
    "Value": [
        n_athletes,
        f"{total_records:,}",
        records_per_athlete_avg,
        sensor_channels,
        n_classes,
        f"{avg_span_s} s",
        "Not provided",
    ],
}).set_index("Property")

display(summary)
summary.to_csv(S1 / "tables" / "dataset_summary.csv")


### Target Class Balance Snapshot

Computes counts and percentages for each motion phase label

In [ ]:
# class distribution
class_counts = df_raw['Event_Label'].value_counts()
class_dist = pd.DataFrame({"count": class_counts, "percent": (class_counts / len(df_raw) * 100).round(2)})
display(class_dist)
class_dist.to_csv(S1 / "tables" / "class_distribution.csv")

ax = class_counts.plot(kind="bar", color="seagreen", figsize=(8, 4))
ax.set(title="Class Distribution", xlabel="Event Label", ylabel="Samples")
plt.tight_layout()
plt.savefig(S1 / "plots" / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close('all')


### Check for Missing Values

Summarizes missing values per column and reports total missing entries across all sensor channels.

In [ ]:
missing_counts = df_raw.isna().sum()
missing_summary = pd.DataFrame({"missing": missing_counts})
display(missing_summary)
missing_summary.to_csv(S1 / "tables" / "missing_values.csv")

total_missing = missing_counts.sum()
if total_missing == 0:
    print("No missing values detected across any channel.")
else:
    print(f"Total missing entries: {total_missing}")
    print("Channels with gaps:", missing_counts[missing_counts > 0].index.tolist())


### Compute Derived Magnitude Signals

Calculate the Euclidean magnitude of 3-axis accelerometer and gyroscope readings.
These provide a single measure of overall intensity without axis decomposition.


In [ ]:
df_raw["Acc_Mag"] = np.sqrt(df_raw["Acc_X"]**2 + df_raw["Acc_Y"]**2 + df_raw["Acc_Z"]**2)
df_raw["Gyro_Mag"] = np.sqrt(df_raw["Gyro_X"]**2 + df_raw["Gyro_Y"]**2 + df_raw["Gyro_Z"]**2)
print("✓ Computed Acc_Mag and Gyro_Mag")

### PER CLASS SUMMARY STATS

In [ ]:
# Per-class summary statistics
sensor_cols = ["Heart_Rate", "Acc_Mag", "Gyro_Mag"]
summary_stats = df_raw.groupby('Event_Label')[sensor_cols].agg(['mean', 'std']).round(2)
display(summary_stats)

In [ ]:
# heart rate over time
plt.figure(figsize=(12, 4))
plt.plot(df_raw['Timestamp'], df_raw['Heart_Rate'], color='darkred')
plt.title('Heart Rate Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Heart Rate (bpm)')
plt.show()

# accelerometer magnitude over time
plt.figure(figsize=(12, 4))
plt.plot(df_raw['Timestamp'], df_raw['Acc_Mag'], color='green', linewidth=0.8)
plt.title('Acceleration Magnitude Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Acceleration (m/s²)')
plt.show()

# gyroscope magnitude over time
plt.figure(figsize=(12, 4))
plt.plot(df_raw['Timestamp'], df_raw['Gyro_Mag'], color='blue', linewidth=0.8)
plt.title('Gyroscope Magnitude Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Angular Velocity (°/s)')
plt.show()



# ANNOTATED SIGNAL EXPLORATION

### Sensor Distributions by Event Label

Boxplots of Heart Rate, Acceleration Magnitude, and Gyroscope Magnitude grouped by motion phase. 
Overlapping distributions suggest phases are harder to discriminate; separated distributions indicate the sensors are informative for classification.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

order = sorted(df_raw['Event_Label'].unique())

sns.boxplot(data=df_raw, x='Event_Label', y='Heart_Rate', order=order, hue='Event_Label', legend=False, ax=axes[0], palette='crest')
axes[0].set_title('Heart Rate by Phase')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df_raw, x='Event_Label', y='Acc_Mag', order=order, hue='Event_Label', legend=False, ax=axes[1], palette='crest')
axes[1].set_title('Acc Magnitude by Phase')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

sns.boxplot(data=df_raw, x='Event_Label', y='Gyro_Mag', order=order, hue='Event_Label', legend=False, ax=axes[2], palette='crest')
axes[2].set_title('Gyro Magnitude by Phase')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Sensor Distributions by Event Label', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(S1 / 'plots' / 'sensor_distributions_by_label.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')


### Annotated Signal Exploration

For each of the 6 motion phase labels, a representative 51-sample window (25 before, 25 after) is extracted from Athlete A001 and plotted across three derived signals: Heart Rate, Acceleration Magnitude, and Gyroscope Magnitude. A vertical dashed line marks the event onset boundary.

In [ ]:
WINDOW  = 25
LABELS  = sorted(df_raw["Event_Label"].unique())
SIGNALS = ["Heart_Rate", "Acc_Mag", "Gyro_Mag"]
COLORS  = {"Heart_Rate": "crimson", "Acc_Mag": "green", "Gyro_Mag": "steelblue"}
YLABELS = {"Heart_Rate": "HR (bpm)", "Acc_Mag": "Acc Mag (g)", "Gyro_Mag": "Gyro Mag (°/s)"}

athlete = df_raw[df_raw["Athlete_ID"] == "A001"].sort_values("Timestamp").reset_index(drop=True)

def plot_signal_grid(labels, title, filename):
    fig, axes = plt.subplots(3, 3, figsize=(15, 9))

    for row, label in enumerate(labels):
        onsets = athlete[
            (athlete["Event_Label"] == label) &
            (athlete["Event_Label"].shift(1) != label)
        ].index
        valid = [i for i in onsets if i >= WINDOW and i + WINDOW < len(athlete)]

        if not valid:
            for col in range(3):
                axes[row, col].text(0.5, 0.5, "No valid window", ha="center", va="center",
                                    transform=axes[row, col].transAxes)
            continue

        transition_idx = valid[0]
        segment = athlete.iloc[transition_idx - WINDOW: transition_idx + WINDOW + 1].copy()
        segment["sample"] = range(-WINDOW, WINDOW + 1)

        for col, signal in enumerate(SIGNALS):
            ax = axes[row, col]
            ax.plot(segment["sample"], segment[signal], color=COLORS[signal], linewidth=1.2)
            ax.axvline(0, color="black", linestyle="--", linewidth=1.0)
            ax.set_ylabel(YLABELS[signal], fontsize=8)
            ax.tick_params(labelsize=7)

            if row == 0:
                ax.set_title(YLABELS[signal], fontsize=10)
            if row == 2:
                ax.set_xlabel("Sample (0 = onset)", fontsize=8)

        axes[row, 0].annotate(
            label, xy=(-0.35, 0.5), xycoords="axes fraction",
            fontsize=9, fontweight="bold", va="center", ha="left", rotation=90
        )

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(S2 / "plots" / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")

plot_signal_grid(
    LABELS[:3],
    "Annotated Signal Exploration: Phases 1-3 (Athlete A001)",
    "annotated_signal_exploration_1.png"
)

plot_signal_grid(
    LABELS[3:],
    "Annotated Signal Exploration: Phases 4-6 (Athlete A001)",
    "annotated_signal_exploration_2.png"
)

print("""
Signal Pattern Analysis:
- Heart Rate: Broadly similar across phases due to synthetic data generation. The average HR per athlete plot reveals individual differences.
- Accelerometer: Magnitude captures overall movement intensity. Box plots show overlapping distributions across phases,motivating the need for feature engineering.
- Gyroscope: Angular velocity is similarly distributed across phases, suggesting rotational patterns require spectral features to differentiate motion classes.
- Annotated event plots show the raw signal context around each labeled motion phase for a single representative athlete.
""")


## PART 2: DATASET PREPROCESSING

### Clean Records and Normalize Labels
- No missing values confirmed.
- Removes duplicate rows.
- Drops invalid timestamps.
- Sorts by athlete and time.
- Normalizes `Event_Label` text for consistent grouping and modeling.

In [ ]:
df_clean = df_raw.copy()

rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep="first")
print(f"Removed {rows_before - len(df_clean)} duplicate rows")

invalid_ts = int(df_clean["Timestamp"].isna().sum())
if invalid_ts:
    print(f"Dropping {invalid_ts} rows with invalid timestamps")
    df_clean = df_clean.dropna(subset=["Timestamp"])

df_clean = df_clean.sort_values(["Athlete_ID", "Timestamp"])

# Reset index once
df_clean = df_clean.reset_index(drop=True)

df_clean["Event_Label"] = (
    df_clean["Event_Label"].astype(str).str.strip().str.lower()
)

#### Missing Value Findings & Interpolation Strategy

**Findings:** No missing values were detected across any of the 7 sensor channels (`Heart_Rate`, `Acc_X`, `Acc_Y`, `Acc_Z`, `Gyro_X`, `Gyro_Y`, `Gyro_Z`), the event label column, or the timestamp column. The dataset is complete as received.

**Interpolation strategy (if gaps were present):**

| Channel | Strategy | Rationale |
|---|---|---|
| `Heart_Rate` | Linear interpolation | HR changes slowly and smoothly; linear fill preserves physiological trend between adjacent readings |
| `Acc_X/Y/Z` | Linear interpolation | Short gaps in accelerometer data are well approximated by linear transitions between samples at 10 Hz |
| `Gyro_X/Y/Z` | Linear interpolation | Same reasoning as accelerometer; gyroscope signals are continuous and slowly varying within a phase |

For all channels, interpolation would be applied within each athlete's session separately to avoid bleeding values across athletes. Gaps longer than **5 consecutive samples (0.5 s at 10 Hz)** would be flagged rather than interpolated, as large gaps indicate sensor dropout rather than noise and could corrupt window-level features downstream.

### Sensor Range Checks

Compares observed numeric ranges with expected physiological and sensor limits.

In [ ]:
numeric_cols = df_clean.select_dtypes(include="number").columns
range_rows = []
for col in numeric_cols:
    range_rows.append({
        "column": col,
       "min": round(df_clean[col].min(), 2),
        "max": round(df_clean[col].max(), 2),
    })
range_df = pd.DataFrame(range_rows).set_index("column").T


LOW_Q = 0.01
HIGH_Q = 0.99
BUFFER = 0.1

SENSOR_THRESHOLDS = {}

for col in numeric_cols:
    low = df_clean[col].quantile(LOW_Q)
    high = df_clean[col].quantile(HIGH_Q)
    span = high - low

    SENSOR_THRESHOLDS[col] = (
        low - BUFFER * span,
        high + BUFFER * span,
    )

display(range_df)

threshold_counts = {}
for col, (low, high) in SENSOR_THRESHOLDS.items():
    if col in df_clean.columns:
        count = int(((df_clean[col] < low) | (df_clean[col] > high)).sum())
        threshold_counts[col] = count

display(pd.Series(threshold_counts, name="out_of_range"))
print("Only a negligible number of outliers (<0.1%) were detected using percentile-based thresholds.")
print("So, no outlier filtering was applied.")

### Sampling Rate Estimation

Computes the inter-sample interval per athlete from the timestamps to confirm the effective sampling rate before any preprocessing decisions.

In [ ]:
# Compute inter-sample intervals per athlete
intervals = (
    df_clean.groupby("Athlete_ID")["Timestamp"]
    .apply(lambda t: t.sort_values().diff().dt.total_seconds().dropna())
)

interval_stats = intervals.groupby(level=0).agg(
    mean_interval=("mean"),
    median_interval=("median"),
    std_interval=("std"),
    min_interval=("min"),
    max_interval=("max"),
)
interval_stats["sampling_rate_hz"] = (1 / interval_stats["median_interval"]).round(2)

display(interval_stats.round(4))
interval_stats.to_csv(S3 / "plots" / "sampling_rate_per_athlete.csv")

overall_median = intervals.median()
overall_hz = round(1 / overall_median, 2)
print(f"\nOverall median inter-sample interval : {overall_median:.4f} s")
print(f"Overall effective sampling rate       : {overall_hz} Hz")


### Signal Preprocessing: Filtering Decision

The dataset was sampled at 10 Hz, giving a Nyquist frequency of 5 Hz. Because the target movements include rapid transitions such as sprint initiation and jump landing, applying a low-pass filter could attenuate short-duration peaks and smooth motion-relevant transients. 


Application of a low-pass Butterworth filter to the 10Hz signals produced no meaningful noise reduction and in some channels introduced signal distortion, widening the amplitude range beyond the raw signal. This confirms that filtering is inappropriate for this sampling rate and the preprocessing pipeline proceeds without it.


In [ ]:
print("No filtering applied, raw signal values retained for all channels.")

### Per Athlete Z score normalization

Each athlete has their own baseline physiology and movement style. Per-athlete z-score means: for each athlete separately, subtract their own mean and divide by their own std for each channel. So every athlete's signals are centered at 0 with unit variance, the model then sees motion shape, not athlete baseline.

This matters most for LOEO, when you train on 4 athletes and test on the 5th, if the test athlete's scale is different from the training athletes, the model struggles. Normalizing per athlete removes that problem.

In [ ]:
NORM_COLS = ["Heart_Rate", "Acc_X", "Acc_Y", "Acc_Z", "Gyro_X", "Gyro_Y", "Gyro_Z", "Acc_Mag", "Gyro_Mag"]

# Snapshot before normalization for plotting
df_before = df_clean[["Athlete_ID", "Heart_Rate", "Acc_Mag"]].copy()
df_before["stage"] = "Before"

# Per-athlete z-score normalization
for col in NORM_COLS:
    df_clean[col] = df_clean.groupby("Athlete_ID")[col].transform(
        lambda x: (x - x.mean()) / x.std()
    )

# Snapshot after normalization
df_after = df_clean[["Athlete_ID", "Heart_Rate", "Acc_Mag"]].copy()
df_after["stage"] = "After"

# Verify: mean ≈ 0 and std ≈ 1 per athlete
verification = df_clean.groupby("Athlete_ID")[NORM_COLS].agg(["mean", "std"]).round(4)
print("Post-normalization check (mean ≈ 0, std ≈ 1 per athlete):")
display(verification)

# Before / after plot
df_compare = pd.concat([df_before, df_after], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color in zip(axes, ["Heart_Rate", "Acc_Mag"], ["crimson", "green"]):
    before = df_compare[df_compare["stage"] == "Before"]
    after  = df_compare[df_compare["stage"] == "After"]

    positions_before = [i * 2     for i in range(5)]
    positions_after  = [i * 2 + 0.8 for i in range(5)]
    athletes = sorted(df_clean["Athlete_ID"].unique())

    bp1 = ax.boxplot(
        [before[before["Athlete_ID"] == a][col].values for a in athletes],
        positions=positions_before, widths=0.6, patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.3),
        medianprops=dict(color="black"), whiskerprops=dict(color=color),
        capprops=dict(color=color), flierprops=dict(marker=".", color=color, alpha=0.2)
    )
    bp2 = ax.boxplot(
        [after[after["Athlete_ID"] == a][col].values for a in athletes],
        positions=positions_after, widths=0.6, patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.8),
        medianprops=dict(color="white"), whiskerprops=dict(color=color),
        capprops=dict(color=color), flierprops=dict(marker=".", color=color, alpha=0.2)
    )

    ax.set_xticks([i * 2 + 0.4 for i in range(5)])
    ax.set_xticklabels(athletes)
    ax.set_title(f"{col}: Before vs After Z-Score")
    ax.set_ylabel("Value")
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ["Before", "After"], loc="upper right")

plt.suptitle("Per-Athlete Z-Score Normalization: Before vs After", fontsize=13)
plt.tight_layout()
plt.savefig(S3 / "plots" / "zscore_normalization_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


> Note: Per-athlete z-score normalization was applied to all channels. While inter-athlete baseline differences were minimal in this dataset, as evidenced by the similar pre-normalization distributions across athletes, normalization is retained as a standard preprocessing step to ensure scale invariance during LOEO evaluation, where the test athlete's signal range must be comparable to training athletes.

## Segmentation Strategy

A sliding window sweep was conducted across 10 window sizes (0.5 s to 5.0 s at 10 Hz) using majority-label assignment and stratified k-fold cross-validation. Analysis of the dataset's event block structure revealed a median consecutive block length of 1 row (0.1 s) and a maximum of 6 rows (0.6 s).

Sliding window analysis showed W=1.0 s as the optimal size and yielding the most stable cross-validation performance. This window size is carried forward for feature extraction.

### Sliding Window Sweep (Evidence)

F1-macro was measured for 10 window sizes (0.5 s–5.0 s) using majority-label assignment to establish whether any window size produces stable classification. 

In [ ]:
WINDOW_SIZES_SW = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]  # seconds, 10 uniform steps

def extract_sliding_windows(df, window_seconds, step_ratio=STEP_RATIO):
    window_delta = pd.to_timedelta(window_seconds, unit="s")
    step_delta   = pd.to_timedelta(window_seconds * step_ratio, unit="s")
    X, y = [], []
    for _, group in df.groupby("Athlete_ID"):
        group        = group.sort_values("Timestamp")
        current_time = group["Timestamp"].min()
        end_time     = group["Timestamp"].max()
        while current_time + window_delta <= end_time:
            win = group[
                (group["Timestamp"] >= current_time) &
                (group["Timestamp"] <  current_time + window_delta)
            ]
            if len(win) > 0:
                data         = win[FEATURE_COLS].values
                chunk_labels = win["Event_Label"].values
                unique, counts = np.unique(chunk_labels, return_counts=True)
                majority = unique[np.argmax(counts)]
                feats = np.concatenate([data.mean(axis=0), data.std(axis=0),
                                        data.min(axis=0),  data.max(axis=0)])
                X.append(feats)
                y.append(majority)
            current_time += step_delta
    return np.array(X), np.array(y)

sw_results = []

for ws in WINDOW_SIZES_SW:
    X, y = extract_sliding_windows(df_clean, ws)
    if len(X) == 0:
        continue
    y_enc    = le.fit_transform(y)
    n_splits = safe_n_splits(y_enc)
    if n_splits < 2:
        continue
    f1 = run_rf_cv(X, y_enc, n_splits)
    sw_results.append({"window_s": ws, "n_windows": len(X),
                       "f1_mean": f1.mean(), "f1_std": f1.std()})

sw_df = pd.DataFrame(sw_results)
display(sw_df.round(4))
sw_df.to_csv(S4 / "tables" / "sliding_window_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sw_df["window_s"], sw_df["f1_mean"], marker="o", color="steelblue", linewidth=2, label="F1-macro (mean)")
ax.fill_between(sw_df["window_s"],
                sw_df["f1_mean"] - sw_df["f1_std"],
                sw_df["f1_mean"] + sw_df["f1_std"],
                alpha=0.15, color="steelblue", label="±1 std")
ax.set_xlabel("Window size (seconds)", fontsize=11)
ax.set_ylabel("F1-macro (stratified k-fold CV)", fontsize=11)
ax.set_title("Sliding Window Sweep — F1-macro vs Window Size", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(S4 / "plots" / "window_size_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


> Note: Window size = 1 chosen

# PART 3: FEATURE ENGINEERING

- Define reusable helper functions for temporal and spectral descriptors.
- Extract per-window features from cleaned and smoothed data.
- Keep only model-ready windows using purity and minimum-sample guards.
- Rank features using train-athlete data to reduce leakage risk.
- Export selected features and diagnostics to support downstream model training.

In [ ]:
df_features_input = df_clean.copy()

### Feature Extraction — 7 Channels × 7 Features

For each of the 7 raw sensor channels, 7 features are extracted per window:

**Time-domain (4):** mean, standard deviation, RMS, zero-crossing rate  
**Frequency-domain (3):** dominant frequency, spectral energy, spectral entropy

Total feature vector: 7 × 7 = **49 features** per window.

In [ ]:
def zero_crossing_rate(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0.0
    centered = x - np.mean(x)
    signs    = np.sign(centered)
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    if signs[0] == 0:
        signs[0] = 1
    crossings = np.sum(signs[:-1] * signs[1:] < 0)
    return float(crossings / (len(x) - 1))


def spectral_features(x, fs):
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0
    centered  = x - np.mean(x)
    power     = np.abs(rfft(centered)) ** 2
    freqs     = rfftfreq(n, d=1.0 / fs)
    dom_freq  = float(freqs[int(np.argmax(power[1:])) + 1]) if len(power) > 1 else 0.0
    spec_energy = float(np.sum(power) / n)
    power_sum   = np.sum(power)
    if power_sum <= 0 or len(power) <= 1:
        spec_entropy = 0.0
    else:
        prob = power / power_sum
        spec_entropy = float(-np.sum(prob * np.log2(prob + 1e-12)) / np.log2(len(prob)))
    return dom_freq, spec_energy, spec_entropy


In [ ]:
def extract_window_features(df, sampling_rate_hz, window_seconds=0.5, step_ratio=0.5, min_samples=1):
    window_delta = pd.to_timedelta(window_seconds, unit="s")
    step_delta   = pd.to_timedelta(max(window_seconds * step_ratio, 0.1), unit="s")
    rows = []

    for athlete_id, athlete_df in df.groupby("Athlete_ID", sort=False):
        athlete_df  = athlete_df.sort_values("Timestamp")
        current_time = athlete_df["Timestamp"].min()
        end_time     = athlete_df["Timestamp"].max()

        while current_time + window_delta <= end_time:
            win_df = athlete_df[
                (athlete_df["Timestamp"] >= current_time) &
                (athlete_df["Timestamp"] <  current_time + window_delta)
            ]

            if len(win_df) >= min_samples:
                label_counts = win_df["Event_Label"].value_counts()
                row = {
                    "Athlete_ID":    athlete_id,
                    "window_start":  current_time,
                    "window_end":    current_time + window_delta,
                    "n_samples":     len(win_df),
                    "Event_Label":   label_counts.index[0],
                    "label_purity":  float(label_counts.iloc[0] / len(win_df)),
                }

                for col in RAW_SENSOR_COLS:       # 7 channels
                    x = win_df[col].to_numpy(dtype=float)
                    dom_freq, spec_energy, spec_entropy = spectral_features(x, sampling_rate_hz)
                    row[f"{col}_mean"]         = float(np.mean(x))
                    row[f"{col}_sd"]           = float(np.std(x, ddof=0))
                    row[f"{col}_rms"]          = float(np.sqrt(np.mean(np.square(x))))
                    row[f"{col}_zcr"]          = zero_crossing_rate(x)
                    row[f"{col}_dom_freq"]     = dom_freq
                    row[f"{col}_spec_energy"]  = spec_energy
                    row[f"{col}_spec_entropy"] = spec_entropy

                rows.append(row)

            current_time += step_delta

    return pd.DataFrame(rows)


### Extract & Save Feature Artifacts

Runs the feature extraction pipeline over the cleaned dataset and writes all outputs to `out/05_features/`:

| File | Contents |
|---|---|
| `features_all.csv` | Full 49-feature table with metadata (athlete, window, label, purity) |
| `features_model.csv` | Model-ready subset — feature columns + label only |
| `feature_columns.txt` | Ordered list of the 49 feature column names |

In [ ]:
df_features = extract_window_features(
    df_clean,
    sampling_rate_hz=FS,
    window_seconds=WINDOW_SECONDS,
    step_ratio=STEP_RATIO,
    min_samples=1,
)

print(f"Windows extracted : {len(df_features):,}")
print(f"Feature columns   : {len([c for c in df_features.columns if any(c.endswith(s) for s in ['_mean','_sd','_rms','_zcr','_dom_freq','_spec_energy','_spec_entropy'])]):,}")
print(f"Classes           : {sorted(df_features['Event_Label'].unique())}")
print(f"Label purity (avg): {df_features['label_purity'].mean():.3f}")
display(df_features.head())

# Feature columns only (no metadata)
FEAT_COLS = [c for c in df_features.columns
             if any(c.endswith(s) for s in
                    ['_mean','_sd','_rms','_zcr','_dom_freq','_spec_energy','_spec_entropy'])]

# Save artifacts
df_features.to_csv(S5 / "tables" / "features_all.csv", index=False)

df_features[FEAT_COLS + ["Event_Label"]].to_csv(S5 / "tables" / "features_model.csv", index=False)

(S5 / "tables" / "feature_columns.txt").write_text("\n".join(FEAT_COLS))

print(f"\nSaved:")
print(f"  {S5 / 'tables' / 'features_all.csv'}")
print(f"  {S5 / 'tables' / 'features_model.csv'}")
print(f"  {S5 / 'tables' / 'feature_columns.txt'}")


### Feature Ranking & Selection

Features are ranked using three complementary criteria and aggregated into a consensus score:

| Criterion | Captures |
|---|---|
| Mutual information | Non-linear dependence between feature and class label |
| ANOVA F-statistic | Linear separability of class means |
| Random Forest importance | Combined non-linear discriminative power |

Each criterion is normalised to [0, 1] and averaged into a consensus rank score. The top N features (target: 20–25 out of 49) are retained. Within the selected set, any feature with Pearson correlation > 0.95 with a higher-ranked feature is dropped to remove redundancy.

In [ ]:
TOP_N      = 20   # target features to keep before correlation pruning
CORR_THRESH = 0.95

X_all  = df_features[FEAT_COLS].values
y_all  = le.fit_transform(df_features["Event_Label"].values)

# --- Three ranking criteria ---
mi    = mutual_info_classif(X_all, y_all, random_state=RANDOM_STATE)
f_stat, _ = f_classif(X_all, y_all)
f_stat = np.nan_to_num(f_stat)

rf_imp = make_rf().fit(X_all, y_all).feature_importances_

# Normalise each to [0, 1] and average
def norm01(v):
    r = v - v.min()
    return r / r.max() if r.max() > 0 else r

consensus = (norm01(mi) + norm01(f_stat) + norm01(rf_imp)) / 3

ranking_df = pd.DataFrame({
    "feature":    FEAT_COLS,
    "mi":         mi,
    "f_stat":     f_stat,
    "rf_imp":     rf_imp,
    "consensus":  consensus,
}).sort_values("consensus", ascending=False).reset_index(drop=True)
ranking_df["rank"] = ranking_df.index + 1

display(ranking_df.round(4))
ranking_df.to_csv(S5 / "tables" / "feature_ranking.csv", index=False)

# --- Select top N ---
top_features = ranking_df["feature"].iloc[:TOP_N].tolist()

# --- Correlation pruning ---
corr_matrix = df_features[top_features].corr().abs()

to_drop = set()
for i, feat_i in enumerate(top_features):
    if feat_i in to_drop:
        continue
    for feat_j in top_features[i + 1:]:
        if feat_j in to_drop:
            continue
        if corr_matrix.loc[feat_i, feat_j] > CORR_THRESH:
            to_drop.add(feat_j)   # drop lower-ranked (feat_j comes after feat_i)

selected_features = [f for f in top_features if f not in to_drop]

print(f"\nTop {TOP_N} by consensus rank  : {len(top_features)} features")
print(f"Dropped (corr > {CORR_THRESH})  : {len(to_drop)}  {sorted(to_drop)}")
print(f"Final selected features       : {len(selected_features)}")
print(f"  {selected_features}")

# Save selected feature set
df_selected = df_features[["Athlete_ID"] + selected_features + ["Event_Label"]]
df_selected.to_csv(S5 / "tables" / "features_selected.csv", index=False)
(S5 / "tables" / "selected_feature_columns.txt").write_text("\n".join(selected_features))
print(f"\nSaved: {S5 / 'tables' / 'features_selected.csv'}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Consensus score bar chart
axes[0].barh(ranking_df["feature"].iloc[:TOP_N][::-1],
             ranking_df["consensus"].iloc[:TOP_N][::-1],
             color="steelblue")
axes[0].set_xlabel("Consensus Score (normalised)")
axes[0].set_title(f"Top {TOP_N} Features by Consensus Rank")

# Correlation heatmap of selected features
corr_sel = df_features[selected_features].corr()
mask = np.triu(np.ones_like(corr_sel, dtype=bool))
sns.heatmap(corr_sel, mask=mask, ax=axes[1], cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=0.4,
            xticklabels=selected_features, yticklabels=selected_features)
axes[1].set_title("Selected Feature Correlation Matrix")
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].tick_params(axis="y", rotation=0,  labelsize=7)

plt.tight_layout()
plt.savefig(S5 / "plots" / "feature_ranking_and_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


# PART 4: MODEL TRAINING AND HYPERPARAMETER TUNING 

This section covers:
- Model selection: five candidate classifiers chosen for small tabular multi-class data
- Training pipeline: StandardScaler → SelectKBest (mutual information) → Classifier
- Cross-validation: 5-Fold Stratified CV and Leave-One-Athlete-Out (LOAO)
- Hyperparameter tuning: GridSearchCV for each model
- Saving all tuned models and a summary table for downstream evaluation

## IMPORTS FOR MODELING

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier

MODEL_DIR = OUT_DIR / "saved_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Modeling imports ready.")
print(f"Models will be saved to: {MODEL_DIR.resolve()}")


### Prepare Feature Matrix, Labels, and Groups

We use the engineered feature matrix produced in Part 3 (`df_selected`). Meta columns are dropped, labels are integer-encoded, and athlete IDs are kept as group labels for Leave-One-Athlete-Out CV.

In [ ]:
META_COLS = ["Athlete_ID", "window_start", "window_end",
             "window_seconds", "step_seconds", "n_samples", "label_purity"]

X      = df_selected.drop(columns=META_COLS + ["Event_Label"], errors="ignore").values
y_raw  = df_selected["Event_Label"].values
y_enc  = le.fit_transform(y_raw)
groups = df_selected["Athlete_ID"].values

print(f"X      : {X.shape}")
print(f"y_enc  : {y_enc.shape}  classes: {list(le.classes_)}")
print(f"groups : {len(groups)}  athletes: {sorted(set(groups))}")


### Train/Test Split & Feature Scaling

A stratified 80/20 split preserves class proportions in both sets. `StandardScaler` is fit on `X_train` only and applied to both splits, this prevents any test-set statistics from leaking into the training pipeline. The scaled arrays `X_train_sc` and `X_test_sc` are the input for all six classical models.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y_enc, groups,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_enc,
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"X_train : {X_train_sc.shape}  |  X_test : {X_test_sc.shape}")
print(f"y_train : {len(y_train)}       |  y_test : {len(y_test)}")
print()
print("Class distribution — train:")
for cls, cnt in zip(*np.unique(y_train, return_counts=True)):
    print(f"  {le.classes_[cls]:>15} : {cnt}")
print("Class distribution — test:")
for cls, cnt in zip(*np.unique(y_test, return_counts=True)):
    print(f"  {le.classes_[cls]:>15} : {cnt}")


### Cross-Validation Strategy

Two CV strategies are used at different stages:

| Strategy | Data | Purpose |
|---|---|---|
| **Stratified 5-fold** | `X_train_sc` / `X_test_sc` | Model comparison — stable F1-macro estimates with preserved class proportions |
| **Leave-One-Athlete-Out (LOSO)** | Full `X`, scaled per fold | Generalisation — train on 4 athletes, test on the 5th unseen athlete |

These are kept strictly separate. The random 80/20 split mixes athletes across train and test, so it cannot be used for LOSO. For LOSO, `StandardScaler` is fit inside each fold on the training athletes only to prevent any test-athlete statistics from leaking into training.

In [ ]:
# --- Strategy 1: Stratified 5-fold (on 80/20 split) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Strategy 1 — Stratified 5-fold")
print(f"  Train : {X_train_sc.shape}  |  Test : {X_test_sc.shape}")
print(f"  Splits: {skf.get_n_splits(X_train_sc, y_train)}")

# --- Strategy 2: Leave-One-Athlete-Out (on full dataset, scaler fit per fold) ---
logo = LeaveOneGroupOut()

# Scale full X once for inspection, but per-fold scaling happens inside model eval loops
scaler_full = StandardScaler()
X_sc        = scaler_full.fit_transform(X)

print()
print("Strategy 2 — Leave-One-Athlete-Out (LOSO)")
print(f"  Full dataset : {X_sc.shape}")
print(f"  Folds        : {logo.get_n_splits(X_sc, y_enc, groups)}")
for fold, (tr, te) in enumerate(logo.split(X_sc, y_enc, groups)):
    held_out = np.unique(groups[te])[0]
    print(f"  Fold {fold+1}: train on {sorted(set(groups[tr]))}  →  test on [{held_out}]  ({len(te)} samples)")


### Baseline: Dummy Classifier

Establishes the performance floor that every real model must beat. With 6 roughly balanced classes, chance accuracy is ~16.7%. A `most_frequent` dummy is evaluated under both CV strategies — any model scoring below these baselines is worse than random guessing.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)

dummy_skf  = cross_val_score(dummy, X_sc, y_enc, cv=skf,  scoring="accuracy")
dummy_logo = cross_val_score(dummy, X_sc, y_enc, cv=logo, scoring="accuracy", groups=groups)

chance = 1 / len(le.classes_)
print(f"Chance level (uniform) : {chance:.3f}")
print(f"Dummy 5-fold  accuracy : {dummy_skf.mean():.3f} ± {dummy_skf.std():.3f}")
print(f"Dummy LOSO    accuracy : {dummy_logo.mean():.3f} ± {dummy_logo.std():.3f}")

DUMMY_BASELINE_SKF  = dummy_skf.mean()
DUMMY_BASELINE_LOSO = dummy_logo.mean()


Model Selection

In [ ]:
candidate_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=600, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "SVM-RBF": SVC(
        kernel="rbf", random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Naive Bayes": GaussianNB(
        var_smoothing=1e-9
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softmax", num_class=6,
        random_state=RANDOM_STATE, eval_metric="mlogloss", verbosity=0
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=300, random_state=RANDOM_STATE
    ),
}

print("Candidate models:")
for name in candidate_models:
    print(f"  • {name}")


### Pre-Tuning Baseline Evaluation

All candidate models are evaluated with default settings under both CV schemes to identify which are worth the additional tuning effort. Scores are compared against the dummy baseline.

In [ ]:
baseline_results = []

for name, clf in candidate_models.items():
    skf_scores  = cross_val_score(clf, X_sc, y_enc, cv=skf,  scoring="accuracy")
    logo_scores = cross_val_score(clf, X_sc, y_enc, cv=logo, scoring="accuracy", groups=groups)

    baseline_results.append({
        "Model":       name,
        "5-Fold Mean": round(skf_scores.mean(),  4),
        "5-Fold Std":  round(skf_scores.std(),   4),
        "LOSO Mean":   round(logo_scores.mean(), 4),
        "LOSO Std":    round(logo_scores.std(),  4),
    })
    print(f"{name:25s} | 5-Fold: {skf_scores.mean():.3f} ± {skf_scores.std():.3f} "
          f"| LOSO: {logo_scores.mean():.3f} ± {logo_scores.std():.3f}")

baseline_df = pd.DataFrame(baseline_results).set_index("Model")
display(baseline_df)
baseline_df.to_csv(S6 / "tables" / "baseline_scores.csv")
print("\nDummy baselines — 5-Fold: {:.3f}  |  LOSO: {:.3f}".format(
    DUMMY_BASELINE_SKF, DUMMY_BASELINE_LOSO))


In [ ]:
models_list = baseline_df.index.tolist()
x = np.arange(len(models_list))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(x - w/2, baseline_df["5-Fold Mean"], w,
       yerr=baseline_df["5-Fold Std"], capsize=4,
       color="steelblue", alpha=0.85, label="5-Fold CV")
ax.bar(x + w/2, baseline_df["LOSO Mean"], w,
       yerr=baseline_df["LOSO Std"], capsize=4,
       color="darkorange", alpha=0.85, label="LOSO")

ax.axhline(DUMMY_BASELINE_SKF,  color="steelblue", linestyle="--", linewidth=1.2,
           label=f"Dummy 5-Fold ({DUMMY_BASELINE_SKF:.2f})")
ax.axhline(DUMMY_BASELINE_LOSO, color="darkorange", linestyle="--", linewidth=1.2,
           label=f"Dummy LOSO ({DUMMY_BASELINE_LOSO:.2f})")

ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=20, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Pre-Tuning Baseline Accuracy — 5-Fold vs LOSO")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(S6 / "plots" / "pretuning_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


### Hyperparameter Tuning

## Random Forest

In [ ]:
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    {
        "n_estimators":     [100, 200, 300, 500],
        "max_depth":        [None, 10, 20, 30],
        "min_samples_leaf": [1, 2, 4],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

rf_gs.fit(X_train_sc, y_train)

print(f"\nBest CV accuracy : {rf_gs.best_score_:.4f}")
print(f"Best params      : {rf_gs.best_params_}")

with open(MODEL_DIR / "random_forest_tuned.pkl", "wb") as f:
    pickle.dump(rf_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — SVM (RBF Kernel)

In [ ]:
svm_gs = GridSearchCV(
    SVC(kernel="rbf", random_state=RANDOM_STATE, class_weight="balanced"),
    {
        "C":     [0.1, 1, 10, 100, 1000],
        "gamma": ["scale", "auto", 0.01, 0.001],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

svm_gs.fit(X_train_sc, y_train)

print(f"\nBest CV accuracy : {svm_gs.best_score_:.4f}")
print(f"Best params      : {svm_gs.best_params_}")

with open(MODEL_DIR / "svm_rbf_tuned.pkl", "wb") as f:
    pickle.dump(svm_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — Decision Tree

In [ ]:
dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    {
        "max_depth":        [3, 5, 10, 15, None],
        "min_samples_leaf": [1, 2, 4, 8],
        "criterion":        ["gini", "entropy"],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

dt_gs.fit(X_train_sc, y_train)

print("Best CV accuracy :", round(dt_gs.best_score_, 4))
print("Best params      :", dt_gs.best_params_)

with open(MODEL_DIR / "decision_tree_tuned.pkl", "wb") as f:
    pickle.dump(dt_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — Naive Bayes

In [ ]:
nb_gs = GridSearchCV(
    GaussianNB(),
    {
        "var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

nb_gs.fit(X_train_sc, y_train)

print("Best CV accuracy :", round(nb_gs.best_score_, 4))
print("Best params      :", nb_gs.best_params_)

with open(MODEL_DIR / "naive_bayes_tuned.pkl", "wb") as f:
    pickle.dump(nb_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — AdaBoost

In [ ]:
ada_gs = GridSearchCV(
    AdaBoostClassifier(random_state=RANDOM_STATE),
    {
        "n_estimators":  [50, 100, 200, 300],
        "learning_rate": [0.01, 0.1, 0.5, 1.0],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

ada_gs.fit(X_train_sc, y_train)

print("Best CV accuracy :", round(ada_gs.best_score_, 4))
print("Best params      :", ada_gs.best_params_)

with open(MODEL_DIR / "adaboost_tuned.pkl", "wb") as f:
    pickle.dump(ada_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — XGBoost

In [ ]:
xgb_gs = GridSearchCV(
    XGBClassifier(
        objective="multi:softmax",
        num_class=len(le.classes_),
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        verbosity=0,
    ),
    {
        "n_estimators":  [100, 200, 300],
        "max_depth":     [3, 5, 7],
        "learning_rate": [0.05, 0.1, 0.2],
        "subsample":     [0.7, 0.85, 1.0],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

xgb_gs.fit(X_train_sc, y_train)

print("Best CV accuracy :", round(xgb_gs.best_score_, 4))
print("Best params      :", xgb_gs.best_params_)

with open(MODEL_DIR / "xgboost_tuned.pkl", "wb") as f:
    pickle.dump(xgb_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning — MLP

In [ ]:
mlp_gs = GridSearchCV(
    MLPClassifier(max_iter=500, random_state=RANDOM_STATE),
    {
        "hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
        "learning_rate_init": [0.001, 0.01],
        "alpha":              [0.0001, 0.001, 0.01],
    },
    cv=skf, scoring="accuracy", n_jobs=-1, refit=True, verbose=1
)

mlp_gs.fit(X_train_sc, y_train)

print("Best CV accuracy :", round(mlp_gs.best_score_, 4))
print("Best params      :", mlp_gs.best_params_)

with open(MODEL_DIR / "mlp_tuned.pkl", "wb") as f:
    pickle.dump(mlp_gs.best_estimator_, f)
print("Model saved.")


### Hyperparameter Tuning Summary

Collects best CV accuracy and best parameters from all tuned models.

In [ ]:
tuned_models = {
    "Random Forest":  rf_gs,
    "SVM-RBF":        svm_gs,
    "Decision Tree":  dt_gs,
    "Naive Bayes":    nb_gs,
    "AdaBoost":       ada_gs,
    "XGBoost":        xgb_gs,
    "MLP":            mlp_gs,
}

summary_rows = []
for name, gs in tuned_models.items():
    summary_rows.append({
        "Model":            name,
        "Best CV Accuracy": round(gs.best_score_, 4),
        "Best Params":      gs.best_params_,
    })

tuning_summary_df = (
    pd.DataFrame(summary_rows)
    .set_index("Model")
    .sort_values("Best CV Accuracy", ascending=False)
)

display(tuning_summary_df)
tuning_summary_df.to_csv(S6 / "tables" / "tuning_summary.csv")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["steelblue" if v > DUMMY_BASELINE_SKF else "salmon"
          for v in tuning_summary_df["Best CV Accuracy"]]
ax.barh(tuning_summary_df.index[::-1], tuning_summary_df["Best CV Accuracy"][::-1],
        color=colors[::-1])
ax.axvline(DUMMY_BASELINE_SKF, color="red", linestyle="--", linewidth=1.2,
           label=f"Dummy baseline ({DUMMY_BASELINE_SKF:.3f})")
ax.set_xlabel("Best CV Accuracy (5-fold)")
ax.set_title("Tuned Model Comparison")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(S6 / "plots" / "tuning_summary.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")
print("Saved tuning_summary.csv and tuning_summary.png")


### Pre-Tuning vs Post-Tuning Accuracy

In [ ]:
pre_lookup = {
    "Random Forest":  baseline_df.loc["Random Forest",  "5-Fold Mean"],
    "SVM-RBF":        baseline_df.loc["SVM-RBF",        "5-Fold Mean"],
    "Decision Tree":  baseline_df.loc["Decision Tree",  "5-Fold Mean"],
    "Naive Bayes":    baseline_df.loc["Naive Bayes",    "5-Fold Mean"],
    "AdaBoost":       baseline_df.loc["AdaBoost",       "5-Fold Mean"],
    "XGBoost":        float("nan"),
    "MLP":            baseline_df.loc["MLP",            "5-Fold Mean"],
}

models_ordered = tuning_summary_df.index.tolist()
pre_vals  = [pre_lookup.get(m, float("nan")) for m in models_ordered]
post_vals = tuning_summary_df["Best CV Accuracy"].tolist()

x = np.arange(len(models_ordered))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, pre_vals,  w, label="Pre-tuning (default)",  color="steelblue", alpha=0.75)
ax.bar(x + w/2, post_vals, w, label="Post-tuning (best CV)", color="seagreen",  alpha=0.85)
ax.axhline(DUMMY_BASELINE_SKF, color="red", linestyle="--", linewidth=1.2,
           label=f"Dummy baseline ({DUMMY_BASELINE_SKF:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(models_ordered, rotation=20, ha="right")
ax.set_ylabel("Accuracy (5-fold CV)")
ax.set_title("Pre-Tuning vs Post-Tuning Accuracy")
ax.set_ylim(0, 0.6)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(S6 / "plots" / "pretuning_vs_posttuning.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")
print("Saved pretuning_vs_posttuning.png")


### Test Set Evaluation

Each tuned model is evaluated on the held-out 20% test set. This is the final unbiased accuracy estimate — the test set was never used during training or tuning.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

test_rows = []
for name, gs in tuned_models.items():
    y_pred = gs.best_estimator_.predict(X_test_sc)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average="macro", zero_division=0)
    test_rows.append({
        "Model":           name,
        "Test Accuracy":   round(acc, 4),
        "Test F1-macro":   round(f1,  4),
    })

test_df = (
    pd.DataFrame(test_rows)
    .set_index("Model")
    .sort_values("Test Accuracy", ascending=False)
)

display(test_df)

# Bar chart — accuracy + F1
x = np.arange(len(test_df))
w = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, test_df["Test Accuracy"], w, label="Accuracy", color="steelblue", alpha=0.85)
ax.bar(x + w/2, test_df["Test F1-macro"], w, label="F1-macro",  color="seagreen",  alpha=0.85)
ax.axhline(DUMMY_BASELINE_SKF, color="red", linestyle="--", linewidth=1.2,
           label=f"Dummy baseline ({DUMMY_BASELINE_SKF:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(test_df.index, rotation=20, ha="right")
ax.set_ylabel("Score")
ax.set_title("Test Set Evaluation — Tuned Models")
ax.set_ylim(0, 0.6)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(S6 / "plots" / "test_set_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# Print classification report for the best model
best_name = test_df["Test Accuracy"].idxmax()
best_pred = tuned_models[best_name].best_estimator_.predict(X_test_sc)
print(f"Classification report — {best_name}:")
print(classification_report(y_test, best_pred, target_names=le.classes_, zero_division=0))


### Unified Model Comparison Table

For each tuned model: key hyperparameter settings, test-set accuracy, precision, recall, and F1-macro.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def key_settings(name, gs):
    p = gs.best_params_
    if name == "Random Forest":
        return f"n_est={p["n_estimators"]}, max_depth={p["max_depth"]}, min_leaf={p["min_samples_leaf"]}"
    elif name == "SVM-RBF":
        return f"C={p["C"]}, gamma={p["gamma"]}"
    elif name == "Hist Gradient Boosting":
        return f"lr={p["learning_rate"]}, max_iter={p["max_iter"]}, max_depth={p["max_depth"]}"
    elif name == "MLP":
        return f"layers={p["hidden_layer_sizes"]}, alpha={p["alpha"]}, lr={p["learning_rate_init"]}"
    elif name == "KNN":
        return f"k={p["n_neighbors"]}, weights={p["weights"]}, metric={p["metric"]}"
    elif name == "Decision Tree":
        return f"max_depth={p["max_depth"]}, min_leaf={p["min_samples_leaf"]}, criterion={p["criterion"]}"
    elif name == "Naive Bayes":
        return f"var_smoothing={p["var_smoothing"]:.0e}"
    elif name == "AdaBoost":
        return f"n_est={p["n_estimators"]}, lr={p["learning_rate"]}"
    elif name == "XGBoost":
        return f"n_est={p["n_estimators"]}, max_depth={p["max_depth"]}, lr={p["learning_rate"]}"
    return str(p)

unified_rows = []
for name, gs in tuned_models.items():
    y_pred = gs.best_estimator_.predict(X_test_sc)
    unified_rows.append({
        "Model":      name,
        "Key Settings": key_settings(name, gs),
        "Accuracy":   round(accuracy_score(y_test, y_pred), 4),
        "Precision":  round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall":     round(recall_score(y_test, y_pred,    average="macro", zero_division=0), 4),
        "F1-macro":   round(f1_score(y_test, y_pred,        average="macro", zero_division=0), 4),
    })

unified_df = (
    pd.DataFrame(unified_rows)
    .set_index("Model")
    .sort_values("F1-macro", ascending=False)
)

display(unified_df)


### Confusion Matrices

Per-model confusion matrices on the held-out test set, saved to `S6/plots/`.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

for name, gs in tuned_models.items():
    y_pred = gs.best_estimator_.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(le.classes_)))

    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=30)
    ax.set_title(f"Confusion Matrix — {name}", fontsize=11)
    plt.tight_layout()

    fname = "cm_" + name.lower().replace(" ", "_").replace("-", "_") + ".png"
    plt.savefig(S6 / "plots" / fname, dpi=150, bbox_inches="tight")
    # plt.show()
    plt.close("all")
    print(f"Saved {fname}")


### Classical Model Results — Final Comparison Table

All models side by side. Best value per metric is highlighted in green.

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1-macro"]

classical_df = unified_df[["Key Settings"] + metric_cols].copy()
classical_df.to_csv(S6 / "tables" / "classical_model_results.csv")
print("Saved classical_model_results.csv\n")

display(classical_df)

print("\nBest per metric:")
for col in metric_cols:
    best_model = classical_df[col].idxmax()
    best_val   = classical_df[col].max()
    print(f"  {col:<12} → {best_model} ({best_val:.4f})")


## PART 5: DEEP LEARNING PIPELINE EXTENSION

The scaled feature matrix is reshaped into three input formats to support different neural architectures:

| Architecture | Input Shape | Notes |
|---|---|---|
| **Feedforward ANN** | `(N, 25)` | Flat feature vector — no reshaping needed |
| **LSTM** | `(N, 1, 25)` | 1 timestep, 25 features per step |
| **1D-CNN** | `(N, 25, 1)` | 25 "timesteps", 1 channel — treats features as a 1D signal |

**Output layer:** 6 units (one per motion phase class), softmax activation  
**Hidden layers (ANN/LSTM):** ReLU activations  
**1D-CNN:** ReLU on conv layers, softmax on dense output  
**Loss:** sparse categorical cross-entropy (integer-encoded labels)

### Input Reshaping & Array Export

Format and save train/test arrays for all three architectures.

In [ ]:
import numpy as np

n_train, n_features = X_train_sc.shape
n_test              = X_test_sc.shape[0]
n_classes           = len(le.classes_)

print("Input shape summary")
print(f"  Features (F)     : {n_features}")
print(f"  Classes          : {n_classes}  → {list(le.classes_)}")
print(f"  Train samples    : {n_train}")
print(f"  Test  samples    : {n_test}")
print()

# --- Feedforward ANN: already (N, F) ---
X_train_ann = X_train_sc                          # (n_train, 25)
X_test_ann  = X_test_sc                           # (n_test,  25)

# --- LSTM: (N, timesteps=1, features) ---
X_train_lstm = X_train_sc[:, np.newaxis, :]       # (n_train, 1, 25)
X_test_lstm  = X_test_sc[:, np.newaxis, :]        # (n_test,  1, 25)

# --- 1D-CNN: (N, features, channels=1) ---
X_train_cnn  = X_train_sc[:, :, np.newaxis]       # (n_train, 25, 1)
X_test_cnn   = X_test_sc[:, :, np.newaxis]        # (n_test,  25, 1)

print("Reshaped arrays:")
print(f"  ANN   train/test : {X_train_ann.shape}  /  {X_test_ann.shape}")
print(f"  LSTM  train/test : {X_train_lstm.shape} /  {X_test_lstm.shape}")
print(f"  1D-CNN train/test: {X_train_cnn.shape}  /  {X_test_cnn.shape}")
print()

# --- Save all arrays ---
arrays = {
    "X_train_ann":  X_train_ann,
    "X_test_ann":   X_test_ann,
    "X_train_lstm": X_train_lstm,
    "X_test_lstm":  X_test_lstm,
    "X_train_cnn":  X_train_cnn,
    "X_test_cnn":   X_test_cnn,
    "y_train":      y_train,
    "y_test":       y_test,
}

for name, arr in arrays.items():
    path = S6 / "tables" / f"{name}.npy"
    np.save(path, arr)
    print(f"  Saved {name}.npy  {arr.shape}")


### Sequence Preparation — ANN Input Pipeline

Confirms and documents every step of the input pipeline before model construction:

| Step | Decision | Detail |
|---|---|---|
| **Normalization** | Per-athlete z-score → global `StandardScaler` | Applied in preprocessing; scaler fit on train only |
| **Label encoding** | Integer (`LabelEncoder`) | Matches classical models; one-hot available if needed |
| **Train/test split** | Stratified 80/20 | Identical split used for classical models |
| **Input tensor** | `(N, 25)` flat | Feedforward ANN; LSTM/CNN variants in reshaped arrays |
| **Output units** | 6 | One per motion phase class |
| **Loss** | Sparse categorical cross-entropy | Works directly with integer labels |

In [ ]:
# Confirm pipeline state
print("=== ANN Input Pipeline Verification ===")
print()
print("1. Normalization")
print(f"   Per-athlete z-score applied in preprocessing")
print(f"   StandardScaler fit on X_train only — no leakage")
print(f"   Train mean ≈ {X_train_sc.mean():.4f}  std ≈ {X_train_sc.std():.4f}")
print(f"   Test  mean ≈ {X_test_sc.mean():.4f}  std ≈ {X_test_sc.std():.4f}")
print()
print("2. Label Encoding")
print(f"   Strategy : integer (LabelEncoder)")
print(f"   Classes  : {list(enumerate(le.classes_))}")
print(f"   One-hot alternative : tf.keras.utils.to_categorical(y, num_classes={len(le.classes_)})")
print()
print("3. Train / Test Split")
print(f"   Strategy : stratified 80/20  (random_state={RANDOM_STATE})")
print(f"   Train    : {X_train_sc.shape[0]} samples")
print(f"   Test     : {X_test_sc.shape[0]} samples")
print(f"   Matches classical model split : YES")
print()
print("4. Input Tensor Shapes")
print(f"   Feedforward ANN : {X_train_ann.shape}   → input_shape=({X_train_ann.shape[1]},)")
print(f"   LSTM            : {X_train_lstm.shape}  → input_shape=(1, {X_train_lstm.shape[2]})")
print(f"   1D-CNN          : {X_train_cnn.shape}   → input_shape=({X_train_cnn.shape[1]}, 1)")
print()
print("5. Output Layer")
print(f"   Units      : {len(le.classes_)}")
print(f"   Activation : softmax")
print(f"   Loss       : sparse_categorical_crossentropy")
print(f"   Metric     : accuracy")


### ANN/MLP — Model Definition & Training

A feedforward neural network trained on the prepared `X_train_ann` arrays. Architecture: two hidden layers (128 → 64) with ReLU, batch normalisation, and dropout for regularisation. Output: 6-unit softmax. Trained with early stopping to prevent overfitting.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

tf.random.set_seed(RANDOM_STATE)

n_features = X_train_ann.shape[1]
n_classes  = len(le.classes_)

def build_ann(input_dim, n_classes, dropout=0.3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

ann_model = build_ann(n_features, n_classes)
ann_model.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = ann_model.fit(
    X_train_ann, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

ann_model.save(MODEL_DIR / "ann_model.keras")
print("Model saved.")


### ANN — Training Curves & Test Evaluation

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history["loss"],     label="Train loss")
axes[0].plot(history.history["val_loss"], label="Val loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["accuracy"],     label="Train acc")
axes[1].plot(history.history["val_accuracy"], label="Val acc")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.suptitle("ANN Training Curves", fontsize=12)
plt.tight_layout()
plt.savefig(S6 / "plots" / "ann_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# Test set evaluation
y_pred_ann = np.argmax(ann_model.predict(X_test_ann, verbose=0), axis=1)

ann_acc  = accuracy_score(y_test, y_pred_ann)
ann_prec = precision_score(y_test, y_pred_ann, average="macro", zero_division=0)
ann_rec  = recall_score(y_test, y_pred_ann,    average="macro", zero_division=0)
ann_f1   = f1_score(y_test, y_pred_ann,        average="macro", zero_division=0)

print(f"ANN Test Accuracy  : {ann_acc:.4f}")
print(f"ANN Test Precision : {ann_prec:.4f}")
print(f"ANN Test Recall    : {ann_rec:.4f}")
print(f"ANN Test F1-macro  : {ann_f1:.4f}")

# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
cm = confusion_matrix(y_test, y_pred_ann, labels=np.arange(n_classes))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=ax, colorbar=True, cmap="Blues", xticks_rotation=30)
ax.set_title("Confusion Matrix — ANN")
plt.tight_layout()
plt.savefig(S6 / "plots" / "cm_ann.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


### ANN vs Classical Models — Comparison

In [ ]:
# Build comparison dataframe: classical + ANN
ann_row = pd.DataFrame([{
    "Key Settings": "layers=(128,64), dropout=0.3, lr=1e-3",
    "Accuracy":  round(ann_acc,  4),
    "Precision": round(ann_prec, 4),
    "Recall":    round(ann_rec,  4),
    "F1-macro":  round(ann_f1,   4),
}], index=pd.Index(["ANN"], name="Model"))

metric_cols = ["Accuracy", "Precision", "Recall", "F1-macro"]
compare_df  = pd.concat([unified_df[["Key Settings"] + metric_cols], ann_row])
compare_df  = compare_df.sort_values("F1-macro", ascending=False)

display(compare_df)
compare_df.to_csv(S6 / "tables" / "ann_vs_classical.csv")
print("Saved ann_vs_classical.csv")

# Bar chart
x = np.arange(len(compare_df))
w = 0.2
fig, ax = plt.subplots(figsize=(13, 5))
for i, (col, color) in enumerate(zip(
        metric_cols, ["steelblue", "darkorange", "seagreen", "mediumpurple"])):
    ax.bar(x + (i - 1.5) * w, compare_df[col], w, label=col, color=color, alpha=0.85)
ax.axhline(DUMMY_BASELINE_SKF, color="red", linestyle="--", linewidth=1,
           label=f"Dummy baseline ({DUMMY_BASELINE_SKF:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(compare_df.index, rotation=20, ha="right")
ax.set_ylabel("Score")
ax.set_title("ANN vs Classical Models — Test Set")
ax.set_ylim(0, 0.7)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(S6 / "plots" / "ann_vs_classical.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")
